In [10]:
# Standard library
import os
import operator
from typing import TypedDict, Literal, Annotated, List

# Environment
from dotenv import load_dotenv

# LangChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Pinecone
from pinecone import Pinecone

# External
from tavily import TavilyClient

### loading all api keys from env file

In [11]:
load_dotenv()

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### setting up llm model embedding model and vector db

In [12]:
# LLM models
llm_small = ChatMistralAI(model="mistral-small-latest", api_key=MISTRAL_API_KEY, temperature=0)
llm_large = ChatMistralAI(model="mistral-large-latest", api_key=MISTRAL_API_KEY, temperature=0)

# Embeddings
embeddings = MistralAIEmbeddings(model="mistral-embed", api_key=MISTRAL_API_KEY)

# Pinecone connection
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("rag-chatbot")

# Pinecone vector store
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text"
)

### defining states

In [58]:
class AgentState(TypedDict):

    # Input
    query_type: Literal["rag", "chat"]
    question: str
    chat_history: List[dict]

    # Query processing
    rewritten_query: str
    
    # multiple query
    all_queries: List[str]

    # Retrieval
    documents: Annotated[List[Document], operator.add]
    top_docs : list[dict]
    retrieval_score: float
    retrieval_retry_count: int

    # Generation
    answer: str
    hallucination_retry_count: int
    hallucination_pass: bool

    # Grading
    answer_score: float

    # Final
    sources: Annotated[List[str], operator.add]
    final_answer: str

### making pdf loader that store pdf on vector db

In [14]:
# making pdf loader that store pdf on vector db
def load_split_pdf(filename: str):
    pdf_path = os.path.join(os.getcwd(), filename)
    print(f"Loading {filename}...")

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"Loaded {len(docs)} pages")

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(docs)
    print(f"Split into {len(chunks)} chunks")

    vectorstore.add_documents(chunks)
    print(f"Done! {len(chunks)} chunks pushed to Pinecone")

In [15]:
# ingest pdf
load_split_pdf("rag.pdf")

Loading rag.pdf...
Loaded 6 pages
Split into 15 chunks
Done! 15 chunks pushed to Pinecone


### Checking query if query belongs to rag or general chat

In [16]:
def query_analyzer(state: AgentState) -> AgentState:
    """Classify query as 'rag' or 'chat'"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a query classifier. Classify the user query into one of two categories:
- 'rag': if the query requires searching documents, PDFs, or uploaded content
- 'chat': if the query is general conversation, greetings, or simple questions

Respond with ONLY one word: either 'rag' or 'chat'. Nothing else."""),
        ("human", "{question}")
    ])

    chain = prompt | llm_small
    result = chain.invoke({"question": state["question"]})
    query_type = result.content.strip().lower()

    # Safety check
    if query_type not in ["rag", "chat"]:
        query_type = "rag"

    print(f"Query type: {query_type}")

    return {"query_type": query_type, "question": state["question"]}

In [36]:
result = query_analyzer({"question": "what is corrective retrieval augmented generation in this pdf?"})
print(result)

Query type: rag
{'query_type': 'rag', 'question': 'what is corrective retrieval augmented generation in this pdf?'}


### llm use our query and make query more useful better so llm understand query properly so we get better retrieval answer

In [37]:
def query_rewriter(state: AgentState) -> AgentState:
    """Rewrite the user query for better retrieval"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert query rewriter for a RAG system.
Rewrite the user's question to be more specific and retrieval-friendly.
Keep it concise - one clear sentence only.
Do NOT add multiple options or examples.
Return ONLY the rewritten query. Nothing else."""),
        ("human", "Original question: {question}")
    ])

    chain = prompt | llm_small
    result = chain.invoke({"question": state["question"]})
    rewritten = result.content.strip()

    print(f"Original: {state['question']}")
    print(f"Rewritten: {rewritten}")

    return {"rewritten_query": rewritten, "question": state["question"]}

In [39]:
result = query_rewriter({"question": "what is corrective retrieval augmented generation? in this pdf?"})
print(result)

Original: what is corrective retrieval augmented generation? in this pdf?
Rewritten: What is the definition and explanation of Corrective Retrieval Augmented Generation (CRAG) in this PDF?
{'rewritten_query': 'What is the definition and explanation of Corrective Retrieval Augmented Generation (CRAG) in this PDF?', 'question': 'what is corrective retrieval augmented generation? in this pdf?'}


### if query type is chat then llm directly answer the question without searching documents

In [40]:
def chat_node(state: AgentState) -> AgentState:
    """Directly answer general chat questions without RAG"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Answer the user's question clearly and concisely."),
        ("human", "{question}")
    ])

    chain = prompt | llm_large
    result = chain.invoke({"question": state["question"]})
    answer = result.content.strip()

    print(f"Chat answer: {answer}")

    return {"final_answer": answer}

In [41]:
result = chat_node({"question": "what is attention mechanism"})
print(result)

Chat answer: The **attention mechanism** is a technique in deep learning that allows models to focus on the most relevant parts of the input data when producing an output. It dynamically weighs the importance of different input elements, improving performance in tasks like:

- **Machine Translation** (e.g., Transformer models)
- **Text Summarization**
- **Image Captioning**
- **Speech Recognition**

### **Key Concepts:**
1. **Query, Key, Value (QKV):**
   - The input is split into *queries* (what to focus on), *keys* (what to compare against), and *values* (actual content).
   - Attention scores are computed by comparing queries and keys (e.g., via dot product).

2. **Attention Scores:**
   - Determine how much each input element contributes to the output.
   - Softmax normalizes these scores to sum to 1.

3. **Weighted Sum:**
   - The output is a weighted sum of *values*, where weights are the attention scores.

### **Types of Attention:**
- **Self-Attention:** Relates different posit

### llm use our rewritten query and make 1 query into 3 query so we can get multiple answer based on that quer

In [42]:
def multi_query_generator(state: AgentState) -> AgentState:
    """Generate multiple query variations for better retrieval"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert at generating query variations for a RAG system.
Generate exactly 3 different variations of the given query.
Each variation should be semantically similar but worded differently.
Return ONLY the 3 queries separated by | character.
Example format: query one | query two | query three
Nothing else."""),
        ("human", "Query: {rewritten_query}")
    ])

    chain = prompt | llm_small
    result = chain.invoke({"rewritten_query": state["rewritten_query"]})

    # Simple clean parsing
    queries = [q.strip() for q in result.content.strip().split("|") if q.strip()]

    # Always include the rewritten query itself
    all_queries = [state["rewritten_query"]] + queries

    print(f"Generated {len(all_queries)} queries:")
    for q in all_queries:
        print(f"  - {q}")

    return {"all_queries": all_queries, "rewritten_query": state["rewritten_query"]}

In [43]:
result = query_analyzer({"question": "what is corrective retrieval augmented generation in this pdf?"})
result.update(query_rewriter(result))
result.update(multi_query_generator(result))
print(result)

Query type: rag
Original: what is corrective retrieval augmented generation in this pdf?
Rewritten: What is the definition and explanation of Corrective Retrieval Augmented Generation (CRAG) as described in this PDF?
Generated 4 queries:
  - What is the definition and explanation of Corrective Retrieval Augmented Generation (CRAG) as described in this PDF?
  - What does Corrective Retrieval Augmented Generation (CRAG) mean and how is it explained in the provided PDF?
  - Can you provide the definition and a detailed explanation of Corrective Retrieval Augmented Generation (CRAG) as outlined in the PDF?
  - How is Corrective Retrieval Augmented Generation (CRAG) defined and explained in the document?
{'query_type': 'rag', 'question': 'what is corrective retrieval augmented generation in this pdf?', 'rewritten_query': 'What is the definition and explanation of Corrective Retrieval Augmented Generation (CRAG) as described in this PDF?', 'all_queries': ['What is the definition and explanat

### now making retriever it fetch all multiple query is generated by the multiple query generator and gives answer

In [44]:
def retrieve_documents(state: AgentState) -> AgentState:
    """Retrieve documents from Pinecone for all queries"""

    all_docs = []
    for query in state["all_queries"]:
        docs = vectorstore.similarity_search(query, k=5)
        all_docs.extend(docs)

    print(f"Retrieved {len(all_docs)} chunks total")

    return {"documents": all_docs, "all_queries": state["all_queries"]}

In [45]:
result.update(retrieve_documents(result))
print(f"Total chunks: {len(result['documents'])}")

Retrieved 20 chunks total
Total chunks: 20


### llm check our retrieved documents are relevant to the question or not and give score

In [46]:
def grade_retrieval(state: AgentState) -> AgentState:
    """Grade the relevance of retrieved documents"""

    docs_sample = state["documents"][:5]
    docs_content = "\n\n".join([doc.page_content[:500] for doc in docs_sample])

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a retrieval grader.
Judge whether the retrieved documents are relevant to answer the user's question.
Give a score from 0 to 5:
- 0-2: Documents are not relevant at all
- 3-5: Documents are relevant and useful

Return ONLY a single number between 0 and 5. Nothing else."""),
        ("human", """Question: {question}

Retrieved Documents:
{documents}""")
    ])

    chain = prompt | llm_small
    result = chain.invoke({
        "question": state["question"],
        "documents": docs_content
    })

    try:
        score = float(result.content.strip())
    except:
        score = 3.0

    print(f"Retrieval score: {score}")

    return {"retrieval_score": score}

In [47]:
result.update(retrieve_documents(result))
result.update(grade_retrieval(result))
print(result["retrieval_score"])

Retrieved 20 chunks total
Retrieval score: 5.0
5.0


### refine node works when retrieval score is less than 3 then it loops back to the query rewriter it loops back max attempts 3 times

In [53]:
def refine_query(state: AgentState) -> AgentState:
    """Refine the query when retrieval score is low"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert at reformulating search queries.
The previous query did not retrieve relevant documents.
Generate a completely different version of the query using different keywords and angle.
Return ONLY the new query. Nothing else."""),
        ("human", """Original question: {question}
Previous query that failed: {rewritten_query}""")
    ])

    chain = prompt | llm_small
    result = chain.invoke({
        "question": state["question"],
        "rewritten_query": state["rewritten_query"]
    })

    new_query = result.content.strip()
    new_retry_count = state["retrieval_retry_count"] + 1

    print(f"Refined query: {new_query}")
    print(f"Retry count: {new_retry_count}")

    return {
        "rewritten_query": new_query,
        "retrieval_retry_count": new_retry_count,
        "documents": [] #clear all chunks in our docus state
    }

In [54]:
result["retrieval_retry_count"] = 0
result.update(refine_query(result))
print(result["rewritten_query"])

Refined query: "Understanding the concept of Corrective Retrieval Augmented Generation (CRAG) as described in the provided PDF"
Retry count: 1
"Understanding the concept of Corrective Retrieval Augmented Generation (CRAG) as described in the provided PDF"


### if max count reached that means vector db has no answer regarding our query so
### then query go to the tavily and search the answer of it

In [55]:

def tavily_search(state: AgentState) -> AgentState:
    """Fallback web search when Pinecone retrieval fails"""

    client = TavilyClient(api_key=TAVILY_API_KEY)

    results = client.search(
        query=state["question"],
        max_results=5
    )

    # Convert Tavily results to Document format
    docs = []
    for r in results["results"]:
        doc = Document(
            page_content=r["content"],
            metadata={"source": r["url"], "title": r.get("title", "")}
        )
        docs.append(doc)

    print(f"Tavily retrieved {len(docs)} web results")

    return {"documents": docs}

In [56]:
result.update(tavily_search(result))
print(f"Total web docs: {len(result['documents'])}")
print(f"First result source: {result['documents'][0].metadata['source']}")

Tavily retrieved 5 web results
Total web docs: 5
First result source: https://openreview.net/pdf?id=JnWJbrnaUE


### now we are going to add reranker node that is going to rearrange in order the the retrival gived answer so get better answer and wrong answer got negleted